In [67]:
import pandas as pd

df_diag = pd.read_csv("data/diagnosticos.csv")
df_mri = pd.read_csv("data/escaner.csv")
df_demo = pd.read_csv("data/demograficos.csv")
df_gene = pd.read_csv("data/genetica.csv")
df_mmse = pd.read_csv("data/mmse.csv")
df_adas = pd.read_csv("data/adas.csv")
df_cdr = pd.read_csv("data/cdr.csv")
diccionario = pd.read_csv("data/diccionario.csv")

## FUNCIONES AUXILIARES

In [68]:
def filtrar_instantes(df):
    visitas_por_paciente = df.groupby("PTID")["VISCODE2"].nunique() # agrupamos por id paciente y contamos cuantos instantes cumplen
    pacientes_completos = visitas_por_paciente[visitas_por_paciente == 3].index # cogemos los indices que tienen 3 instantes
    res = df[df["PTID"].isin(pacientes_completos)].copy() # guardamos los pacientes indexados

    return res

def resumen(df):
    print(f"Número de pacientes que tienen los 3 instantes temporales: {df["PTID"].nunique()}")
    print(f"Total de muestras (3*número de pacientes): {len(df)}")
    print(f"Distribución en el instante 'bl': \n{df[df["VISCODE2"] == "bl"]["DIAGNOSIS"].value_counts()}")
    print(f"Distribución en el instante 'mes 12': \n{df[df["VISCODE2"] == "m12"]["DIAGNOSIS"].value_counts()}")
    print(f"Distribución en el instante 'mes 24': \n{df[df["VISCODE2"] == "m24"]["DIAGNOSIS"].value_counts()}")

diccionario = diccionario[diccionario["TBLNAME"].isin(["ADAS", "MMSE", "CDR", "APOERES"])]

## PREPROCESADO DE LA TABLA DE DIAGNÓSTICOS

In [69]:
print(df_diag.columns)
print(df_diag.DIAGNOSIS.unique())

Index(['ORIGPROT', 'COLPROT', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE',
       'DIAGNOSIS', 'DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1',
       'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6', 'DXMDUE',
       'DXMOTHET', 'DXDSEV', 'DXDDUE', 'DXAD', 'DXAPP', 'DXAPROB', 'DXAPOSS',
       'DXPARK', 'DXPDES', 'DXPCOG', 'DXPATYP', 'DXDEP', 'DXOTHDEM', 'DXODES',
       'DXCONFID', 'ID', 'SITEID', 'USERDATE', 'USERDATE2',
       'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR',
       'update_stamp'],
      dtype='str')
<StringArray>
['CN', 'Dementia', 'MCI', nan]
Length: 4, dtype: str


In [70]:
print(df_diag.DIAGNOSIS.value_counts())
print(df_diag.VISCODE2.value_counts())

DIAGNOSIS
MCI         6565
CN          6275
Dementia    2996
Name: count, dtype: int64
VISCODE2
bl      3011
sc      2839
m12     1831
m06     1616
m24     1556
m36      942
m48      829
m72      455
m60      438
m18      349
m84      343
m96      224
m78      159
m108     142
m66      132
m120     117
m90      111
m132      77
m30       77
m54       76
m144      71
m156      67
m102      57
m42       49
m138      48
m150      44
m114      43
m126      36
m162      34
m168      22
m180      16
m174      13
m222       8
m216       7
m186       6
m228       6
m204       5
m234       5
m192       4
m210       4
uns1       1
m198       1
Name: count, dtype: int64


In [71]:
# borramos repetidos
df_diag = df_diag.dropna(subset=["DIAGNOSIS"]) # quitamos nulos de diagnosticos ya que no existe registro
df_diag["VISCODE2"] = df_diag["VISCODE2"].replace({"sc": "bl"})
df_diag["DIAGNOSIS"] = df_diag["DIAGNOSIS"].replace({"Dementia": "AD"})
df_diag = df_diag.drop_duplicates(subset=["PTID", "VISCODE2"]) # eliminamos duplicados del mismo paciente en la misma visita

## 3 INSTANTES (UNO CADA AÑO): BL, MES 12, MES 24

In [72]:
instantes = ["bl", "m12", "m24"]

df_filtrado = df_diag[df_diag["VISCODE2"].isin(instantes)].copy() # filtramos los que estan en estos insantescn

df_final = filtrar_instantes(df_filtrado)
resumen(df_final)

Número de pacientes que tienen los 3 instantes temporales: 1352
Total de muestras (3*número de pacientes): 4056
Distribución en el instante 'bl': 
DIAGNOSIS
MCI    745
CN     419
AD     188
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
MCI    651
CN     430
AD     271
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
MCI    542
CN     430
AD     380
Name: count, dtype: int64


### FILTRAR POR RESONANCIAS DE 3 TESLAS

In [73]:
print(df_mri.columns) # la tabla contiene todos los pacientes que han tenido resonancias de 3 teslas
print(df_mri.VISCODE2.value_counts())

Index(['ORIGPROT', 'COLPROT', 'PTID', 'RID', 'VISCODE', 'VISCODE2',
       'FIELD_STRENGTH', 'RECNO', 'MMCONDCT', 'MMREASON', 'EXAMDATE',
       'MMSCOUT', 'MMSMPRAGE', 'MMRMPRAGE', 'MMB1HEAD', 'MMB1BODY', 'MMECHO',
       'MMPHAN', 'MMTRNSFR', 'MMTRNDATE', 'MMTRNDATEYR_DRVD', 'MMARCHIVE',
       'MMLPDONE', 'MMLPINTER', 'SEDATIVE', 'OTRDEV', 'MOTION', 'EYEKEEP',
       'EYEASK', 'EYEKEEP4', 'EYEASK4', 'EYEKEEP6', 'EYEASK6', 'MALFUNC',
       'FLDSTRNGTH', 'ID', 'SITEID', 'USERDATE', 'USERDATE2',
       'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR',
       'update_stamp'],
      dtype='str')
VISCODE2
sc       1569
m12      1205
scmri    1023
m24      1008
m06       914
m03       789
m48       485
m36       332
m72       229
bl        208
m84       150
m60       129
m78       123
m96       112
m66       101
m18        96
m90        92
m30        73
m132       65
m120       61
m108       61
m54        58
m156       50
m102       50
m144       44
m138       38
m42        37
m150

In [74]:
df_mri["VISCODE2"] = df_mri["VISCODE2"].replace({"sc": "bl"})
df_mri = df_mri.drop_duplicates(subset=["PTID", "VISCODE2"])

df_3t = pd.merge( # join con el df anterior y nos quedamos con pacientes que estan en ambos df
    df_final,
    df_mri[["PTID", "VISCODE2", "FIELD_STRENGTH"]],
    on=["PTID", "VISCODE2"],
    how="inner"
)

df_3t = filtrar_instantes(df_3t)
resumen(df_3t)

Número de pacientes que tienen los 3 instantes temporales: 275
Total de muestras (3*número de pacientes): 825
Distribución en el instante 'bl': 
DIAGNOSIS
MCI    137
CN      86
AD      52
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
MCI    117
CN      85
AD      73
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
AD     95
MCI    93
CN     87
Name: count, dtype: int64


### FILTRADO POR DATOS DEMOGRÁFICOS

In [75]:
print(df_demo.columns)

Index(['ORIGPROT', 'COLPROT', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE',
       'PTSOURCE', 'PTGENDER', 'PTDOB', 'PTDOBYY', 'PTHAND', 'PTMARRY',
       'PTEDUCAT', 'PTWORKHS', 'PTWORK', 'PTNOTRT', 'PTRTYR', 'PTHOME',
       'PTTLANG', 'PTPLANG', 'PTADBEG', 'PTCOGBEG', 'PTADDX', 'PTETHCAT',
       'PTRACCAT', 'PTIDENT', 'PTORIENT', 'PTORIENTOT', 'PTENGSPK', 'PTNLANG',
       'PTENGSPKAGE', 'PTCLANG', 'PTLANGSP', 'PTLANGWR', 'PTSPTIM',
       'PTSPOTTIM', 'PTLANGPR1', 'PTLANGSP1', 'PTLANGRD1', 'PTLANGWR1',
       'PTLANGUN1', 'PTLANGPR2', 'PTLANGSP2', 'PTLANGRD2', 'PTLANGWR2',
       'PTLANGUN2', 'PTLANGPR3', 'PTLANGSP3', 'PTLANGRD3', 'PTLANGWR3',
       'PTLANGUN3', 'PTLANGPR4', 'PTLANGSP4', 'PTLANGRD4', 'PTLANGWR4',
       'PTLANGUN4', 'PTLANGPR5', 'PTLANGSP5', 'PTLANGRD5', 'PTLANGWR5',
       'PTLANGUN5', 'PTLANGPR6', 'PTLANGSP6', 'PTLANGRD6', 'PTLANGWR6',
       'PTLANGUN6', 'PTLANGTTL', 'PTETHCATH', 'PTASIAN', 'PTOPI', 'PTBORN',
       'PTBIRPL', 'PTIMMAGE', 'PTIMMWHY', 'PTBIR

In [76]:
df_demo = df_demo.drop_duplicates(subset=["PTID"])

df_3t_demo = pd.merge(
    df_3t,
    df_demo[["PTID", "PTGENDER", "PTDOB"]],
    on=["PTID"],
    how="inner"
)

df_3t_demo = filtrar_instantes(df_3t_demo)
resumen(df_3t_demo)

Número de pacientes que tienen los 3 instantes temporales: 275
Total de muestras (3*número de pacientes): 825
Distribución en el instante 'bl': 
DIAGNOSIS
MCI    137
CN      86
AD      52
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
MCI    117
CN      85
AD      73
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
AD     95
MCI    93
CN     87
Name: count, dtype: int64


### FILTRADO GENÉTICO

Cada persona hereda dos copias de este gen, una de cada progenitor, lo que da lugar a diferentes combinaciones (genotipos) basadas en tres variantes o alelos principales: ε2 (protector), ε3 (neutral) y ε4 (riesgo)

- APOE 3/3 (E3/E3): Es el genotipo más común (presente en más de la mitad de la población) y se considera el "punto de referencia". Indica un riesgo promedio o neutro de desarrollar Alzheimer.
- APOE 3/4 (E3/E4): Indica un riesgo aumentado. Tener una copia del alelo ε4 triplica aproximadamente las probabilidades de desarrollar la enfermedad en comparación con el tipo 3/3.
- APOE 4/4 (E4/E4): Es el perfil de mayor riesgo genético. Tener dos copias de ε4 aumenta el riesgo entre 8 y 15 veces y se asocia con una aparición de síntomas a una edad más temprana.
- APOE 2/3 (E2/E3): Se considera una combinación que ofrece cierta protección. El alelo ε2 parece reducir el riesgo y, en caso de desarrollar la enfermedad, esta suele aparecer más tarde en la vida.
- APOE 2/4 (E2/E4): Es una combinación poco frecuente. Aunque el alelo ε2 es protector y el ε4 es de riesgo, el riesgo general suele ser intermedio, situándose por encima del 3/3 pero por debajo del 3/4.
- APOE 2/2 (E2/E2): Es el genotipo más protector y el menos común (solo en un 1-2% de la población). Se asocia con el menor riesgo posible de Alzheimer relacionado con este gen.

In [77]:
print(df_gene.columns)
print(df_gene.GENOTYPE.value_counts())

Index(['ORIGPROT', 'COLPROT', 'PTID', 'RID', 'VISCODE', 'GENOTYPE', 'APTESTDT',
       'APVOLUME', 'APRECEIVE', 'APAMBTEMP', 'APRESAMP', 'APUSABLE', 'ID',
       'SITEID', 'USERDATE', 'USERDATE2', 'update_stamp'],
      dtype='str')
GENOTYPE
3/3    1417
3/4    1016
4/4     275
2/3     219
2/4      71
2/2      10
Name: count, dtype: int64


In [78]:
df_gene = df_gene.drop_duplicates(subset=["PTID"])

df_3t_demo_gene = pd.merge(
    df_3t_demo,
    df_gene[["PTID", "GENOTYPE"]],
    on=["PTID"],
    how="inner"
)

df_3t_demo_gene = filtrar_instantes(df_3t_demo_gene)
resumen(df_3t_demo_gene)

Número de pacientes que tienen los 3 instantes temporales: 274
Total de muestras (3*número de pacientes): 822
Distribución en el instante 'bl': 
DIAGNOSIS
MCI    136
CN      86
AD      52
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
MCI    116
CN      85
AD      73
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
AD     95
MCI    92
CN     87
Name: count, dtype: int64


## UNIÓN DE TABLAS DE PRUEBAS COGNITIVAS

- ADAS: test específico más utilizado en ensayos clínicos para medir el deterioro de la memoria y la capacidad cognitiva en el Alzheimer.
    - Variables: 
        - TOTSCORE (ADAS-11): Versión estándar de 11 tareas (puntuación de 0 a 70).
        - TOT13 (ADAS-13): Versión extendida de 13 tareas, añade pruebas de memoria de trabajo y cancelación (puntuación de 0 a 85).

    - ¿Qué significa la puntuación?: A más puntos, PEOR. Mide "errores" o "faltas".
        - Cerca de 0: Cerebro perfecto, sin errores.
        - Puntuaciones altas: Deterioro grave (mide la progresión del Alzheimer).
- CDR: escala global basada en una entrevista al paciente y a su familia. Evalúa 6 áreas funcionales (memoria, orientación, juicio, comunidad, hogar y cuidado personal).
    - Variables: 
        - CDGLOBAL (Puntuación Global): Clasifica al paciente en un grado definitivo (Valores: 0, 0.5, 1, 2, 3).
        - CDRSB (Sum of Boxes / Suma de Casillas): Suma las notas de las 6 áreas por separado (puntuación de 0 a 18). Es mucho más sensible y precisa para ver cambios pequeños.
    - ¿Qué significa la puntuación?: A más puntos, PEOR.
        - 0: Sano / Normal.
        - 0.5: Deterioro Cognitivo Leve (MCI) / Dudoso.
        - 1: Demencia Leve (Alzheimer inicial).
        - 2: Demencia Moderada.
        - 3: Demencia Grave.
- MMSE: test rápido de cribado general (orientación, memoria, atención, lenguaje). 
    - Variable: MMSCORE 
    - Rango: 0 a 30. A más puntos, MEJOR. 
        - 24 a 30: Cognitivamente normal. 
        - Menos de 24: Sospecha de deterioro/demencia (cuantos menos puntos, más grave).

In [79]:
print(df_adas.VISCODE2.value_counts())
print(df_mmse.VISCODE2.value_counts())
print(df_cdr.VISCODE2.value_counts())

VISCODE2
bl      3023
m12     1791
m06     1616
m24     1557
m36      934
m48      831
m72      447
m60      432
m18      347
m84      306
m96      218
m78      147
m108     143
m66      122
m120     116
m90      100
m30       78
m132      77
m54       73
m144      61
m156      61
m102      59
m138      45
m42       44
m114      43
m150      38
m126      36
m162      26
m180      16
m168      15
m174      11
m186       7
m216       6
m222       6
m204       5
m192       4
m210       4
m228       4
m234       3
uns1       1
m198       1
Name: count, dtype: int64
VISCODE2
sc      4337
m12     1793
m06     1617
m24     1556
m36      937
m48      833
m72      448
m60      432
f        407
m18      347
m84      305
m96      217
m78      146
m108     144
m66      123
m120     117
m90      100
m132      77
m30       77
m54       73
m144      61
m156      61
m102      59
m138      45
m42       44
m114      43
m150      38
m126      36
m162      26
m180      16
m168      15
m174      11
m186   

In [80]:
df_cdr["VISCODE2"] = df_cdr["VISCODE2"].replace({"sc": "bl"})
df_mmse["VISCODE2"] = df_mmse["VISCODE2"].replace({"sc": "bl"})

df_adas = df_adas.drop_duplicates(subset=["PTID", "VISCODE2"])
df_cdr = df_cdr.drop_duplicates(subset=["PTID", "VISCODE2"])
df_mmse = df_mmse.drop_duplicates(subset=["PTID", "VISCODE2"])

df_adas = df_adas.dropna(subset=["TOTSCORE", "TOTAL13"])
df_cdr = df_cdr.dropna(subset=["CDRSB", "CDGLOBAL"])
df_mmse = df_mmse.dropna(subset=["MMSCORE"])

df_adas = df_adas[["PTID", "VISCODE2", "TOTSCORE", "TOTAL13"]]
df_cdr = df_cdr[["PTID", "VISCODE2", "CDRSB", "CDGLOBAL"]]
df_mmse = df_mmse[["PTID", "VISCODE2", "MMSCORE"]]

df_pruebas = pd.merge(
    df_adas,
    df_cdr,
    on=["PTID", "VISCODE2"],
    how="inner"
)

df_pruebas = pd.merge(
    df_pruebas,
    df_mmse,
    on=["PTID", "VISCODE2"],
    how="inner"
)

## DATAFRAME FINAL (DIAGNÓSTICO+ESCÁNER+GENÉTICA+DEMOGRAFÍA+PRUEBAS)

In [81]:
df_3t_demo_gene_pruebas = pd.merge(
    df_3t_demo_gene,
    df_pruebas,
    on=["PTID", "VISCODE2"],
    how="inner"
)

df_3t_demo_gene_pruebas = filtrar_instantes(df_3t_demo_gene_pruebas)
resumen(df_3t_demo_gene_pruebas)

Número de pacientes que tienen los 3 instantes temporales: 256
Total de muestras (3*número de pacientes): 768
Distribución en el instante 'bl': 
DIAGNOSIS
MCI    128
CN      83
AD      45
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
MCI    109
CN      82
AD      65
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
MCI    88
AD     85
CN     83
Name: count, dtype: int64


In [82]:
df_3t_demo_gene_pruebas[df_3t_demo_gene_pruebas["PTID"] == "002_S_1018"]

,ORIGPROT,COLPROT,PTID,RID,VISCODE,VISCODE2,EXAMDATE,DIAGNOSIS,DXNORM,DXNODEP,...,update_stamp,FIELD_STRENGTH,PTGENDER,PTDOB,GENOTYPE,TOTSCORE,TOTAL13,CDRSB,CDGLOBAL,MMSCORE
113,ADNI1,ADNI1,002_S_1018,1018,bl,bl,2006-12-27,AD,NaN,NaN,...,2007-01-02,3 Tesla,Female,03/1936,3/3,9.00,17.00,3.0,0.5,26.0
270,ADNI1,ADNI1,002_S_1018,1018,m12,m12,2007-12-07,AD,NaN,NaN,...,2008-01-04,3 Tesla,Female,03/1936,3/3,8.33,16.33,3.5,0.5,29.0
412,ADNI1,ADNI1,002_S_1018,1018,m24,m24,2009-01-02,AD,NaN,NaN,...,2009-01-05,3 Tesla,Female,03/1936,3/3,12.00,21.00,3.5,0.5,27.0


In [83]:
print(df_3t_demo_gene_pruebas["PTID"].drop_duplicates().tolist())

['100_S_0015', '023_S_0030', '023_S_0031', '100_S_0035', '023_S_0058', '023_S_0061', '023_S_0139', '032_S_0187', '136_S_0184', '136_S_0195', '082_S_0304', '023_S_0331', '023_S_0376', '131_S_0384', '005_S_0324', '136_S_0196', '027_S_0307', '127_S_0260', '023_S_0388', '037_S_0303', '002_S_0413', '131_S_0441', '027_S_0403', '027_S_0404', '136_S_0299', '018_S_0369', '131_S_0457', '005_S_0448', '136_S_0300', '133_S_0433', '068_S_0473', '136_S_0426', '100_S_0190', '116_S_0382', '032_S_0479', '133_S_0488', '133_S_0493', '018_S_0335', '031_S_0568', '033_S_0514', '068_S_0401', '023_S_0604', '116_S_0392', '018_S_0425', '037_S_0501', '133_S_0525', '067_S_0290', '018_S_0450', '126_S_0605', '136_S_0429', '002_S_0559', '068_S_0478', '127_S_0622', '023_S_0625', '136_S_0579', '053_S_0507', '032_S_0677', '005_S_0553', '005_S_0572', '005_S_0602', '018_S_0633', '012_S_0689', '130_S_0449', '033_S_0733', '033_S_0739', '126_S_0606', '002_S_0729', '033_S_0724', '033_S_0725', '116_S_0649', '116_S_0752', '133_

100_S_0015, 023_S_0030, 023_S_0031, 100_S_0035, 023_S_0058, 023_S_0061, 023_S_0139, 032_S_0187, 136_S_0184, 136_S_0195, 082_S_0304, 023_S_0331, 023_S_0376, 131_S_0384, 005_S_0324, 136_S_0196, 027_S_0307, 127_S_0260, 023_S_0388, 037_S_0303, 002_S_0413, 131_S_0441, 027_S_0403, 027_S_0404, 136_S_0299, 018_S_0369, 131_S_0457, 005_S_0448, 136_S_0300, 133_S_0433, 068_S_0473, 136_S_0426, 100_S_0190, 116_S_0382, 032_S_0479, 133_S_0488, 133_S_0493, 018_S_0335, 031_S_0568, 033_S_0514, 068_S_0401, 023_S_0604, 116_S_0392, 018_S_0425, 037_S_0501, 133_S_0525, 067_S_0290, 018_S_0450, 126_S_0605, 136_S_0429, 002_S_0559, 068_S_0478, 127_S_0622, 023_S_0625, 136_S_0579, 053_S_0507, 032_S_0677, 005_S_0553, 005_S_0572, 005_S_0602, 018_S_0633, 012_S_0689, 130_S_0449, 033_S_0733, 033_S_0739, 126_S_0606, 002_S_0729, 033_S_0724, 033_S_0725, 116_S_0649, 116_S_0752, 133_S_0638, 016_S_0769, 021_S_0753, 031_S_0830, 027_S_0835, 023_S_0916, 005_S_0814, 133_S_0792, 133_S_0771, 033_S_0920, 033_S_0922, 023_S_0926, 023_S_0963, 068_S_0802, 130_S_0886, 130_S_0505, 130_S_0956, 133_S_0912, 133_S_0727, 127_S_0844, 033_S_1016, 023_S_1046, 133_S_1031, 002_S_0954, 012_S_1009, 033_S_1098, 033_S_1116, 031_S_1066, 033_S_1086, 013_S_1035, 023_S_1126, 016_S_1117, 002_S_1018, 130_S_0969, 002_S_1070, 133_S_1055, 032_S_1169, 067_S_0607, 016_S_1121, 133_S_1170, 023_S_1190, 128_S_1148, 012_S_1212, 007_S_1206, 007_S_1222, 016_S_1138, 023_S_1247, 031_S_1209, 136_S_1227, 116_S_1249, 027_S_1081, 094_S_1241, 116_S_1232, 002_S_1261, 023_S_1262, 033_S_1279, 033_S_1309, 100_S_1286, 131_S_1301, 082_S_1256, 002_S_1280, 100_S_1226, 012_S_1321, 068_S_0872, 020_S_1288, 027_S_1387, 027_S_1385, 016_S_1326, 002_S_1268, 094_S_1293, 094_S_1267, 007_S_1304, 052_S_1250, 052_S_1251, 037_S_0588, 131_S_1389, 068_S_1075, 051_S_1123, 051_S_1131, 051_S_1331, 067_S_1253, 027_S_6002, 002_S_6007, 022_S_6013, 024_S_6033, 130_S_6047, 301_S_6056, 941_S_6068, 130_S_6072, 014_S_6076, 037_S_6083, 135_S_6110, 135_S_6104, 116_S_6133, 141_S_6116, 014_S_6145, 141_S_6178, 020_S_6185, 070_S_6236, 099_S_6175, 007_S_6255, 033_S_6298, 301_S_6297, 019_S_6315, 007_S_6341, 033_S_6352, 305_S_6378, 127_S_6241, 041_S_6401, 135_S_6389, 099_S_6396, 018_S_6414, 114_S_6347, 129_S_6452, 135_S_6446, 035_S_6480, 019_S_6483, 035_S_6488, 033_S_6497, 168_S_6467, 127_S_6436, 020_S_6513, 135_S_6509, 135_S_6510, 135_S_6544, 116_S_6550, 127_S_6433, 168_S_6541, 127_S_6512, 033_S_6572, 941_S_6575, 941_S_6580, 135_S_6586, 127_S_6549, 168_S_6591, 130_S_6604, 114_S_6597, 168_S_6619, 019_S_6635, 130_S_6612, 099_S_6632, 168_S_6634, 027_S_6643, 023_S_6661, 019_S_6668, 131_S_6616, 033_S_6705, 126_S_6724, 941_S_6254, 019_S_6757, 305_S_6742, 305_S_6744, 129_S_6784, 022_S_6796, 131_S_6805, 016_S_6816, 305_S_6810, 129_S_6830, 016_S_6839, 027_S_6842, 941_S_6854, 129_S_6852, 129_S_6857, 057_S_6869, 305_S_6871, 033_S_6889, 035_S_6927, 035_S_6947, 033_S_6969, 033_S_6976, 035_S_6999, 035_S_7000, 126_S_7015, 019_S_7016, 131_S_7032, 126_S_7060, 035_S_7073, 126_S_7083, 033_S_7088, 035_S_7105, 035_S_7121, 941_S_10002, 941_S_10003, 941_S_10011, 033_S_10005

## Dudas
1. minimo de datos que necesitamos
1. si pasa algo si los intervalos entre imagenes varian entre pacientes. Si uso para unos (0, 6, 12 meses) y para otros (0, 12, 24 meses)
1. teniendo el cuenta el anterior, podemos juntar resonancias en el dataset, por ejemplo para el mismo paciente guardar el trio 0-6-18 y por ejemplo 0-12-24
1. hay dos tipos de formatos en la imagen NIFTL y DICOM, cual uso y que programa me recomiendas para visualizarlos (aun no he hecho nada respecto la imagen, solo he buscado los pacientes que hay)
1. Hay resonancias T1 y T2 cual usamos y cual es la diferencia
1. Existen MRI (LO QUE HE USADO) y PET cual usamos y cual es la diferencia
1. Hay dos tipos de resonancias, de 1.5 teslas y 3 teslas. Pasa algo si usamos ambas para tener mas imagenes?
1. Cuando voy a buscar las imagenes por el id del paciente, se puede buscar por imagenes raw (sin procesar) y procesadas. Cual es la diferencia y cual elijo (supongo que las procesadas no?)
1. A la hora de buscar imagenes lo he filtrado por T1, 3d, 3t (CONTINUACION ABAJO)

## PACIENTES

Para comprobar con otros pacientes hay que obtener primero los ids, descargar el csv y actualizarlo

In [84]:
df_pacientes = pd.read_csv("data/PACIENTES.csv")

In [85]:
print(df_pacientes.columns)
print(df_pacientes["Visit"].value_counts())

Index(['Image Data ID', 'Subject', 'Group', 'Sex', 'Age', 'Visit', 'Modality',
       'Description', 'Type', 'Acq Date', 'Format', 'Downloaded'],
      dtype='str')
Visit
m24       344
m12       336
m06       263
m36       207
bl        202
y1        178
y2        175
sc        174
m18       156
4_init     60
y3         53
y4         39
4_m12      31
init       31
v21        24
v06        15
v11        14
v41        11
v31        11
4_m24      10
y5         10
m48         7
v51         6
4_sc        4
nv          4
Name: count, dtype: int64


In [86]:
print(df_pacientes["Description"].unique())

<StringArray>
[ 'Accelerated Sagittal IR-FSPGR (MSV21)',
    'Accelerated Sagittal MPRAGE (MSV21)',
    'Accelerated Sagittal MPRAGE (MSV22)',
             'HS Sagittal MPRAGE (MSV22)',
             'CS Sagittal MPRAGE (MSV22)',
       'Accelerated Sag IR-FSPGR (MSV21)',
 'Sagittal 3D Accelerated MPRAGE (MSV21)',
                       'Calibration Scan',
 'Sagittal 3D Accelerated 0 angle MPRAGE',
            'Axial Field Mapping 0 angle',
            'Accelerated Sagittal MPRAGE',
               'Accelerated Sag IR-FSPGR',
  'Accelerated Sagittal MPRAGE Phase A-P',
                   'AAHead_Scout_MPR_tra',
                   'AAHead_Scout_MPR_cor',
                   'AAHead_Scout_MPR_sag',
                           'AAHead_Scout',
                           'Cal 8HRBRAIN',
          'Accelerated Sagittal IR-FSPGR',
         'Sagittal 3D Accelerated MPRAGE',
                    'Axial Field Mapping',
     'Accelerated Sagittal MPRAGE REPEAT',
         'Accelerated Sagittal MPRAGE_ND

In [87]:
df_pacientes = df_pacientes.rename(columns={
    "Subject": "PTID",
    "Visit": "VISCODE2",
    "Image Data ID": "IMAGEUID"
})

df_pacientes["VISCODE2"] = df_pacientes["VISCODE2"].replace({
    "sc": "bl", 
    "init": "bl", 
    "4_init": "bl", 
    "4_sc": "bl",
    
    "y1": "m12", 
    "4_m12": "m12", 
    "v11": "m12",
    
    "y2": "m24", 
    "4_m24": "m24", 
    "v21": "m24"
})

print(df_pacientes["VISCODE2"].value_counts())

VISCODE2
m12    559
m24    553
bl     471
m06    263
m36    207
m18    156
y3      53
y4      39
v06     15
v41     11
v31     11
y5      10
m48      7
v51      6
nv       4
Name: count, dtype: int64


In [88]:
df_pacientes = df_pacientes[df_pacientes["VISCODE2"].isin(instantes)]
df_pacientes = df_pacientes[df_pacientes["Description"].str.lower().str.contains("mprage|mp-rage|ir-fspgr|3dtfe")]
df_pacientes = df_pacientes.drop_duplicates(subset=["PTID", "VISCODE2"], keep="first")

df_validos = pd.merge(
    df_3t_demo_gene_pruebas,
    df_pacientes[["PTID", "VISCODE2", "IMAGEUID"]], 
    on=["PTID", "VISCODE2"], 
    how="inner"
)

df_validos = filtrar_instantes(df_validos)
resumen(df_validos)

Número de pacientes que tienen los 3 instantes temporales: 143
Total de muestras (3*número de pacientes): 429
Distribución en el instante 'bl': 
DIAGNOSIS
MCI    70
CN     51
AD     22
Name: count, dtype: int64
Distribución en el instante 'mes 12': 
DIAGNOSIS
MCI    61
CN     52
AD     30
Name: count, dtype: int64
Distribución en el instante 'mes 24': 
DIAGNOSIS
MCI    55
CN     51
AD     37
Name: count, dtype: int64
